In [1]:
import os

import numpy as np
import pandas as pd

import librosa
import soundfile as sf
from scipy import signal
import warnings

from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from pathlib import Path
from transformers import ASTModel, ASTFeatureExtractor,  get_cosine_schedule_with_warmup
from tqdm import tqdm


warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


In [2]:
class_labels = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')['primary_label'].tolist()

test_soundscape_path =  '/kaggle/input/competitions/birdclef-2026/test_soundscapes/'   
# "/kaggle/input/competitions/birdclef-2026/train_soundscapes"    # '/kaggle/input/competitions/birdclef-2026/test_soundscapes/'
test_soundscapes = [os.path.join(test_soundscape_path, afile) for afile in sorted(os.listdir(test_soundscape_path)) if afile.endswith('.ogg')]


# Load AST model: =======>>>>> XXX <<<<=======

In [3]:
class ASTClassifier_OnlyTrainable_Head(nn.Module): 
    def __init__(self, backbone, embed_dim, n_classes): 
        super().__init__()
        self.backbone = backbone 
        
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim), 
            nn.Dropout(0.2),
            nn.Linear(embed_dim, 512), 
            nn.GELU(), 
            nn.Dropout(0.1), 
            nn.Linear(512, n_classes)
            
        )

    def forward(self, input_values): 
        out = self.backbone(input_values)
        return self.head(out.pooler_output)
        

In [4]:
class ASTClassifier_FullTrainable_Model(nn.Module): 
    def __init__(self, backbone, embed_dim, n_classes):
        
        super().__init__()
        self.backbone = backbone
        
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.2),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, n_classes),
        )

    def forward(self, input_values):
        out = self.backbone(input_values=input_values)
        return self.head(out.pooler_output)   # pooler_output: (B, 768)

    def freeze_backbone(self): 
        for p in self.backbone.parameters(): 
            p.requires_grad = False

    def unfreeze_backbone(self): # unfreeze parametes for full fine-tuning
        for p in self.backbone.parameters: 
            p.requires_grad = True

    def unfreeze_last_n_layers(self, n=4): 
        for p in self.backbone.parameters(): 
            p.requires_grad = False

        # now unfreze last n layer + final layer Norm layer of AST
        encLayer = self.backbone.encoder.layer
        totalEncLayer = len(encLayer)

        for layer in encLayer[totalEncLayer - n:]: 
            for p in layer.parameters(): 
                p.requires_grad = True

        for p in self.backbone.layernorm.parameters(): 
            p.requires_grad = True

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f"Unfroze last {n}/{totalEncLayer} encoder layers")
        print(f"Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M params  ({100*trainable/total:.1f}%)")


    
    def get_optimizer(self, strategy="differential", lr_head=1e-3, lr_backbone=1e-5):
        
        if strategy == "head_only": 
            self.freeze_backbone()
            self.torch.optim.AdamW(
                filter(lambda p: p.requires_grad, self.parameters()),
                lr=lr_head, weight_decay=1e-4
            )
            
        elif strategy == "full":
            self.unfreeze_backbone()
            return torch.optim.AdamW(
                self.parameters(), lr=lr_backbone, weight_decay=1e-4
            )

        elif strategy == "differential":
            self.unfreeze_backbone()
            # Different parameter groups → different learning rates
            return torch.optim.AdamW([
                {"params": self.backbone.parameters(), "lr": lr_backbone},  # tiny
                {"params": self.head.parameters(),     "lr": lr_head},      # large
            ], weight_decay=1e-4)

        else:
            raise ValueError(f"Unknown strategy: {strategy}")


In [5]:
def load_ast_offline(n_classes=234, freeze_backbone=False, trainMethod="outerLayerTrainable"): # loaded pretrained weights 
    AST_DIR = "/kaggle/input/datasets/niazmahmud0201/offlinebard-dataset/offline_weights/ast-audioset"

    if not Path(AST_DIR).exists(): 
        raise FileNotFoundError(f"AST pre-trained model not found in {AST_DIR}")

    print("AST model founded and start loading AST model!!!")
    
    feature_extractor = ASTFeatureExtractor.from_pretrained(AST_DIR)
    backbone_ast = ASTModel.from_pretrained(AST_DIR)


    if freeze_backbone==True: 
        for p in backbone_ast.parameters(): 
            p.require_grad = False
        print(f"AST  model backBone Frized")

    
    if trainMethod == "outerLayerTrainable": 
        model = ASTClassifier_OnlyTrainable_Head(
            backbone = backbone_ast,
            embed_dim = backbone_ast.config.hidden_size, # 768
            n_classes = n_classes # 234
        )
    else: 
        model = ASTClassifier_FullTrainable_Model(
            backbone = backbone_ast,
            embed_dim = backbone_ast.config.hidden_size, # 768
            n_classes = n_classes # 234
        )

        

    print(f"Loaded AST model with Parameter:{sum(p.numel() for p in model.parameters()) / 1e6}")
    return model , feature_extractor



device = "cuda" if torch.cuda.is_available() else "cpu"
myAST_model , AST_Feature_extractor =  load_ast_offline(n_classes=234, freeze_backbone=False, trainMethod="FullTrainableModel")

# myAST_model = myAST_model.to(device)

AST model founded and start loading AST model!!!


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: /kaggle/input/datasets/niazmahmud0201/offlinebard-dataset/offline_weights/ast-audioset
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded AST model with Parameter:86.70257


In [6]:
WEIGHTS = "/kaggle/input/datasets/niazmahmud0201/bird-trainedmodel/my_trainedModelWeights/ast_birdclef_weights.pth"
myAST_model.load_state_dict(
    torch.load(WEIGHTS, map_location=device)
)

myAST_model  = nn.DataParallel(myAST_model,  device_ids=[0, 1])
myAST_model = myAST_model.to(device).eval()
print("Model ready")

Model ready


# Now predict with the model 

predictions = pd.DataFrame(columns=['row_id'] + class_labels)

chunks = []
fileName_with_sec = []

for ipdsx, soundscape in enumerate(test_soundscapes):
    sig, rate = librosa.load(path=soundscape, sr=16000) # sec * 16000 -> rate =16000

    for i in range(0, len(sig), rate*5):
        chunk = sig[i:i+rate*5]

        if len(chunk) < rate * 5:
            chunk = np.pad(chunk, (0, rate * 5 - len(chunk)))
        chunks.append(chunk)

    
    # Make predictions for each chunk
    for i, chunk in enumerate(chunks):
        fileName_with_sec.append(os.path.basename(soundscape).split('.')[0] + f'_{i * 5 + 5}')


    if len(chunks)>200: 
        print(ipdsx)
        # now predict with the chunk with the loaded model
        inputs = AST_Feature_extractor(
            chunks,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True,
        )
    
    
        with torch.no_grad(): 
            logits = myAST_model(inputs['input_values'].to(device))
            probs  = torch.sigmoid(logits).cpu().numpy()
    
        
        for row_id, prob in zip(fileName_with_sec, probs):
            new_row = pd.DataFrame([[row_id] + prob.tolist()], columns=['row_id'] + class_labels)
            predictions = pd.concat([predictions, new_row], ignore_index=True)

        
        chunks = []
        fileName_with_sec = []







if len(chunks)>0: 
     # now predict with the chunk with the loaded model
    inputs = AST_Feature_extractor(
        chunks,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True,
    )
    
    
    with torch.no_grad(): 
        logits = myAST_model(inputs['input_values'].to(device))
        probs  = torch.sigmoid(logits).cpu().numpy()
    
        
    for row_id, prob in zip(fileName_with_sec, probs):
        new_row = pd.DataFrame([[row_id] + prob.tolist()], columns=['row_id'] + class_labels)
        predictions = pd.concat([predictions, new_row], ignore_index=True)


In [7]:
def load_and_resample(path, target_sr=16000): # scipy signal is 3x faster than librosa
    sig, sr = sf.read(path, dtype='float32')
    if sig.ndim > 1:
        sig = sig.mean(axis=1)  
    if sr != target_sr:
        sig = signal.resample_poly(sig, target_sr, sr).astype(np.float32)
    return sig

In [8]:
BATCH_SIZE = 200  
rows = []  # append all df data here

all_chunks = []
all_row_ids = []

for soundscape in tqdm(test_soundscapes, desc="Inference"):
    stem = os.path.basename(soundscape).split('.')[0]
    # sig, rate = librosa.load(path=soundscape, sr=16000)
    sig = load_and_resample(soundscape, target_sr=16000)
    rate = 16000
    
    #  now spkit the audio file in 5se*16000 chunk .. cause sr=16000
    for i in range(0, 60 * 16000, 16000 * 5):   # always 12 windows -> 20//5
        
        chunk = sig[i : i + 16000 * 5]
        if len(chunk) < 16000 * 5:
            chunk = np.pad(chunk, (0, 16000 * 5 - len(chunk)))
            
        all_chunks.append(chunk)
        all_row_ids.append(f"{stem}_{(i // (16000*5))*5 + 5}")



    # run the mdel with batched audio file
    if len(all_chunks) >= BATCH_SIZE:
        inputs = AST_Feature_extractor(
            all_chunks, 
            sampling_rate=16000,
            return_tensors="pt", 
            padding=True,
        )
        with torch.no_grad():
            logit = myAST_model(inputs['input_values'].to(device))
            probs = torch.sigmoid(logit).cpu().numpy() # (BATCH_SIZE, 234)

        for row_id, prob in zip(all_row_ids, probs):
            rows.append([row_id] + prob.tolist())

        all_chunks = [] 
        all_row_ids = []


# handle the remaining audio 5s chinks
if len(all_chunks) > 0:
    inputs = AST_Feature_extractor(
        all_chunks, 
        sampling_rate=16000,
        return_tensors="pt", 
        padding=True,
    )
    with torch.no_grad():
        logit =  myAST_model(inputs['input_values'].to(device))
        probs = torch.sigmoid(logit).cpu().numpy()

    for row_id, prob in zip(all_row_ids, probs):
        rows.append([row_id] + prob.tolist())



Inference: 0it [00:00, ?it/s]


In [9]:
predictions = pd.DataFrame(rows, columns=['row_id'] + class_labels)

In [10]:
predictions.to_csv('submission.csv', index=False)
print(f"Saved — shape: {predictions.shape}")
predictions.head()

Saved — shape: (0, 235)


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,22973,22983,22985,23150,23154,23158,23176,23724,24279,24285,24287,24321,244024,25073,25092,25214,326272,41970,43435,47144,47158son01,47158son02,47158son03,47158son04,47158son05,47158son06,47158son07,47158son08,47158son09,47158son10,47158son11,47158son12,47158son13,47158son14,47158son15,47158son16,47158son17,47158son18,47158son19,47158son20,47158son21,47158son22,47158son23,47158son24,47158son25,476521,516975,517063,555123,555145,555146,64898,65377,65380,66971,67107,67252,70711,738183,74113,74580,760266,ashgre1,astcra1,bafcur1,baffal1,banana,barant1,batbel1,baymac,bbwduc,bcwfin2,bkcdon,bkhpar,blchaw1,blheag1,blttit1,bncfly,bobfly1,brcmar1,brnowl,bucmot4,bucpar,bufpar,bunibi1,burowl,camfli1,chacha1,chbmoc1,chobla1,chvcon1,cibspi1,coffal1,compau,compot1,crbthr1,crebec1,dwatin1,epaori4,eulfly1,fabwre1,fepowl,ficman1,flawar1,fotfly,fusfly1,gilhum1,giwrai1,glteme1,grasal3,greani1,greant1,greela,grekis,grepot1,gretho2,greyel,grfdov1,grhtan1,gycwor1,horscr1,houspa,hyamac1,larela1,lesela1,lesgrf1,limpki,linwoo1,litcuc2,litnig1,mabpar,magant1,magtan2,masgna1,nacnig1,ocecra1,oliwoo1,orbtro3,orwpar,osprey,pabspi1,palhor3,paltan1,phecuc1,picpig2,pirfly1,plasla1,platyr1,plcjay1,pluibi1,purjay1,pvttyr1,ragmac1,rebscy1,recfin1,redjun,relser1,rinkin1,rivwar1,roahaw,rubthr1,rufcac2,rufcas2,rufgna3,rufhor2,rufnig1,ruftho1,ruftof1,rumfly1,ruther1,rutjac1,sabspa1,saffin,saytan1,scadov1,schpar1,scther1,shcfly1,shshaw,shtnig1,sibtan2,smbani,smbtin1,sobcac1,sobtyr1,socfly1,sofspi1,souant1,soulap1,souscr1,spbant3,spispi1,sptnig1,squcuc1,stbwoo2,strcuc1,strher2,strowl1,swthum1,swtman1,tattin1,thlwre1,toctou1,trokin,trsowl,undtin1,varant1,watjac1,wesfie1,wfwduc1,whbant2,whbwar2,whiwoo1,whlspi1,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
